# 10 A Dead Unit Blocks the Input Weights

| | |
|---|---|
| **Path** | Multilayer perceptron |
| **Prerequisite** | 09 |

Replace the identity by ReLU, and start the hidden unit on its negative side.

$$
x = 1,\quad y = 1,\quad w = -1,\quad b = 0,\quad v = 1,\quad c = 0
$$

$$
z = -1, \qquad h = \mathrm{ReLU}(-1) = 0, \qquad s = 0
$$

$$
L = \dfrac{1}{2}(0 - 1)^{2} = \dfrac{1}{2}
$$

The backward pass still reaches the hidden unit:

$$
\dfrac{\partial L}{\partial s} = -1, \qquad
\dfrac{\partial L}{\partial h} = -1
$$

The ReLU slope at $z = -1$ is $0$, so

$$
\dfrac{\partial L}{\partial z} = 0, \qquad
\dfrac{\partial L}{\partial w} = 0, \qquad
\dfrac{\partial L}{\partial b} = 0
$$

The input weight is frozen. The output bias is not, because it does not pass through the ReLU:

$$
\dfrac{\partial L}{\partial c} = -1
$$

A step $\eta = 1/2$ moves only $c$,

$$
c \leftarrow 0 - \dfrac{1}{2}(-1) = \dfrac{1}{2}
$$

The new score is $1/2$, and

$$
L = \dfrac{1}{2}\left(\dfrac{1}{2} - 1\right)^{2} = \dfrac{1}{8}
$$

The loss fell. The weight $w$ is still $-1$, so the unit is still dead.


## Example 1 — The slope multiplies the whole input gradient by zero


In [1]:
# The dead ReLU freezes w and b. The output bias still receives -1.
from fractions import Fraction

x, y = 1, 1
w, b, v, c = -1, 0, 1, 0
z = w * x + b
h = z if z > 0 else 0
slope = 1 if z > 0 else 0
s = v * h + c
d_s = s - y
d_h = d_s * v
d_z = d_h * slope
d_w = d_z * x
d_c = d_s
print(z, h, d_s, d_w, d_c)
assert z == -1 and h == 0 and slope == 0
assert d_s == -1
assert d_w == 0 and d_c == -1


-1 0 -1 0 -1


## Example 2 — After the step the unit is still off

$$
z = (-1)(1) + 0 = -1
$$

ReLU still returns $0$. The new loss $1/8$ came entirely from $c = 1/2$.


In [2]:
# c moves to 1/2. w stays -1, so the hidden unit remains off and the loss is 1/8.
from fractions import Fraction

w, c = -1, Fraction(1, 2)
z = w * 1
h = z if z > 0 else 0
s = h + c
loss = Fraction(1, 2) * (s - 1) ** 2
print(w, z, s, loss)
assert w == -1 and z == -1 and h == 0
assert s == Fraction(1, 2)
assert loss == Fraction(1, 8)


-1 -1 1/2 1/8


## Pitfall — A falling loss can hide a frozen feature

The output bias found an easier way to reduce the residual: add $1/2$ after the dead unit. The input weight received no signal, so a later point that needed this unit to notice $x$ still faces $w = -1$.


## Summary

On the negative side of ReLU the slope is $0$, and the input weights of that unit do not change. The output bias skips the bend, so it can still move. Here it cuts the loss from $1/2$ to $1/8$ while $w$ stays $-1$.
